In [1]:
import json
import numpy as np
from scipy import stats
from pathlib import Path

base = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models\2_Logistic_Regression_model\results")

with open(base / "results_lr_baseline_multiseed.json") as f:
    lr_no = json.load(f)

with open(base / "results_lr_network_multiseed.json") as f:
    lr_with = json.load(f)

METRICS = [
    "HitRate@1", "HitRate@5", "HitRate@10", "HitRate@20",
    "NDCG@1",    "NDCG@5",    "NDCG@10",    "NDCG@20",
    "MRR",
]

ALPHA = 0.05 

def extract(per_seed_list, metric):
    return np.array([s[metric] for s in per_seed_list])

def run_tests(no_data, with_data, split="val"):
    key = f"per_seed_{split}"
    results = []
    for metric in METRICS:
        a = extract(no_data[key],   metric)   
        b = extract(with_data[key], metric)   
        diff = b - a                           

        # Paired t-test (n=3, so df=2)
        t_stat, p_val = stats.ttest_rel(b, a)

        results.append({
            "metric":   metric,
            "mean_no":  a.mean(),
            "mean_with":b.mean(),
            "diff":     diff.mean(),
            "diff_std": diff.std(ddof=1),
            "t":        t_stat,
            "p":        p_val,
            "sig":      p_val < ALPHA,
        })
    return results

def print_table(results, title):
    print(f"\n{'═'*78}")
    print(f"  {title}")
    print(f"{'═'*78}")
    header = f"{'Metric':<14} {'no_net':>8} {'with_net':>9} {'Δ (with−no)':>12} {'t':>7} {'p':>8}  sig?"
    print(header)
    print("─" * 78)
    for r in results:
        flag  = "✓ YES" if r["sig"] else "  no"
        delta_sign = "+" if r["diff"] >= 0 else ""
        print(
            f"{r['metric']:<14}"
            f" {r['mean_no']:>8.4f}"
            f" {r['mean_with']:>9.4f}"
            f" {delta_sign}{r['diff']:>+10.4f}"
            f" {r['t']:>7.3f}"
            f" {r['p']:>8.4f}"
            f"  {flag}"
        )
    print("─" * 78)
    n_sig = sum(r["sig"] for r in results)
    print(f"  Significant at α={ALPHA}: {n_sig}/{len(results)} metrics")

for split in ("val", "test"):
    print_table(run_tests(lr_no, lr_with, split), f"Logistic Regression — {split.upper()}")

print("\n\nEffect sizes (Cohen's d) for significant results:")
print(f"{'Model':<14} {'Split':<6} {'Metric':<14} {'Cohen d':>9}  interpretation")
print("─" * 65)

def cohens_d(no_data, with_data, split, metric):
    key = f"per_seed_{split}"
    a = extract(no_data[key], metric)
    b = extract(with_data[key], metric)
    diff = b - a
    return diff.mean() / diff.std(ddof=1)

def interpret(d):
    ad = abs(d)
    if ad < 0.2:  return "negligible"
    if ad < 0.5:  return "small"
    if ad < 0.8:  return "medium"
    return "large"

for model_name, no_data, with_data in [
    ("LR", lr_no, lr_with),
]:
    for split in ("val", "test"):
        res = run_tests(no_data, with_data, split)
        for r in res:
            if r["sig"]:
                d = cohens_d(no_data, with_data, split, r["metric"])
                print(f"{model_name:<14} {split:<6} {r['metric']:<14} {d:>9.3f}  {interpret(d)}")


══════════════════════════════════════════════════════════════════════════════
  Logistic Regression — VAL
══════════════════════════════════════════════════════════════════════════════
Metric           no_net  with_net  Δ (with−no)       t        p  sig?
──────────────────────────────────────────────────────────────────────────────
HitRate@1        0.1616    0.7749 +   +0.6134 1068.797   0.0000  ✓ YES
HitRate@5        0.5689    0.9168 +   +0.3479 766.137   0.0000  ✓ YES
HitRate@10       0.7480    0.9452 +   +0.1972 312.887   0.0000  ✓ YES
HitRate@20       0.8434    0.9679 +   +0.1244 425.383   0.0000  ✓ YES
NDCG@1           0.1616    0.7749 +   +0.6134 1068.797   0.0000  ✓ YES
NDCG@5           0.3669    0.8554 +   +0.4886 2772.879   0.0000  ✓ YES
NDCG@10          0.4257    0.8647 +   +0.4390 3032.001   0.0000  ✓ YES
NDCG@20          0.4499    0.8704 +   +0.4205 3002.435   0.0000  ✓ YES
MRR              0.3366    0.8409 +   +0.5044 1727.467   0.0000  ✓ YES
────────────────────────────

A paired t-test across three random seeds showed that the LR model with network features significantly outperformed the baseline LR model on all ranking metrics for both validation and test splits (p < 0.05).

In [3]:
import json
import numpy as np
from scipy import stats
from pathlib import Path

base = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models\3_Random_forest_model\results")

with open(base / "results_rf_baseline_multiseed.json") as f:
    lr_no = json.load(f)

with open(base / "results_rf_network_multiseed.json") as f:
    lr_with = json.load(f)

METRICS = [
    "HitRate@1", "HitRate@5", "HitRate@10", "HitRate@20",
    "NDCG@1",    "NDCG@5",    "NDCG@10",    "NDCG@20",
    "MRR",
]

ALPHA = 0.05 

def extract(per_seed_list, metric):
    return np.array([s[metric] for s in per_seed_list])

def run_tests(no_data, with_data, split="val"):
    key = f"per_seed_{split}"
    results = []
    for metric in METRICS:
        a = extract(no_data[key],   metric)   
        b = extract(with_data[key], metric)   
        diff = b - a                           

        # Paired t-test (n=3, so df=2)
        t_stat, p_val = stats.ttest_rel(b, a)

        results.append({
            "metric":   metric,
            "mean_no":  a.mean(),
            "mean_with":b.mean(),
            "diff":     diff.mean(),
            "diff_std": diff.std(ddof=1),
            "t":        t_stat,
            "p":        p_val,
            "sig":      p_val < ALPHA,
        })
    return results

def print_table(results, title):
    print(f"\n{'═'*78}")
    print(f"  {title}")
    print(f"{'═'*78}")
    header = f"{'Metric':<14} {'no_net':>8} {'with_net':>9} {'Δ (with−no)':>12} {'t':>7} {'p':>8}  sig?"
    print(header)
    print("─" * 78)
    for r in results:
        flag  = "✓ YES" if r["sig"] else "  no"
        delta_sign = "+" if r["diff"] >= 0 else ""
        print(
            f"{r['metric']:<14}"
            f" {r['mean_no']:>8.4f}"
            f" {r['mean_with']:>9.4f}"
            f" {delta_sign}{r['diff']:>+10.4f}"
            f" {r['t']:>7.3f}"
            f" {r['p']:>8.4f}"
            f"  {flag}"
        )
    print("─" * 78)
    n_sig = sum(r["sig"] for r in results)
    print(f"  Significant at α={ALPHA}: {n_sig}/{len(results)} metrics")

for split in ("val", "test"):
    print_table(run_tests(lr_no, lr_with, split), f"Random Forest — {split.upper()}")

print("\n\nEffect sizes (Cohen's d) for significant results:")
print(f"{'Model':<14} {'Split':<6} {'Metric':<14} {'Cohen d':>9}  interpretation")
print("─" * 65)

def cohens_d(no_data, with_data, split, metric):
    key = f"per_seed_{split}"
    a = extract(no_data[key], metric)
    b = extract(with_data[key], metric)
    diff = b - a
    return diff.mean() / diff.std(ddof=1)

def interpret(d):
    ad = abs(d)
    if ad < 0.2:  return "negligible"
    if ad < 0.5:  return "small"
    if ad < 0.8:  return "medium"
    return "large"

for model_name, no_data, with_data in [
    ("RF", lr_no, lr_with),
]:
    for split in ("val", "test"):
        res = run_tests(no_data, with_data, split)
        for r in res:
            if r["sig"]:
                d = cohens_d(no_data, with_data, split, r["metric"])
                print(f"{model_name:<14} {split:<6} {r['metric']:<14} {d:>9.3f}  {interpret(d)}")


══════════════════════════════════════════════════════════════════════════════
  Random Forest — VAL
══════════════════════════════════════════════════════════════════════════════
Metric           no_net  with_net  Δ (with−no)       t        p  sig?
──────────────────────────────────────────────────────────────────────────────
HitRate@1        0.6204    0.9587 +   +0.3383  92.325   0.0001  ✓ YES
HitRate@5        0.7775    0.9723 +   +0.1948 168.439   0.0000  ✓ YES
HitRate@10       0.8347    0.9788 +   +0.1442 125.295   0.0001  ✓ YES
HitRate@20       0.8991    0.9869 +   +0.0878 214.187   0.0000  ✓ YES
NDCG@1           0.6204    0.9587 +   +0.3383  92.325   0.0001  ✓ YES
NDCG@5           0.7056    0.9659 +   +0.2604 154.264   0.0000  ✓ YES
NDCG@10          0.7240    0.9681 +   +0.2441 155.701   0.0000  ✓ YES
NDCG@20          0.7403    0.9701 +   +0.2298 151.276   0.0000  ✓ YES
MRR              0.6966    0.9656 +   +0.2691 124.855   0.0001  ✓ YES
────────────────────────────────────────

In [2]:
import json
import numpy as np
from scipy import stats
from pathlib import Path

base = Path(r"C:\Users\cieci\OneDrive\Dokumenty\GitHub\bachelor.project\models\4_XGBoost_model\results")

with open(base / "results_xgb_baseline_multiseed.json") as f:
    lr_no = json.load(f)

with open(base / "results_xgb_network_multiseed.json") as f:
    lr_with = json.load(f)

METRICS = [
    "HitRate@1", "HitRate@5", "HitRate@10", "HitRate@20",
    "NDCG@1",    "NDCG@5",    "NDCG@10",    "NDCG@20",
    "MRR",
]

ALPHA = 0.05 

def extract(per_seed_list, metric):
    return np.array([s[metric] for s in per_seed_list])

def run_tests(no_data, with_data, split="val"):
    key = f"per_seed_{split}"
    results = []
    for metric in METRICS:
        a = extract(no_data[key],   metric)   
        b = extract(with_data[key], metric)   
        diff = b - a                           

        # Paired t-test (n=3, so df=2)
        t_stat, p_val = stats.ttest_rel(b, a)

        results.append({
            "metric":   metric,
            "mean_no":  a.mean(),
            "mean_with":b.mean(),
            "diff":     diff.mean(),
            "diff_std": diff.std(ddof=1),
            "t":        t_stat,
            "p":        p_val,
            "sig":      p_val < ALPHA,
        })
    return results

def print_table(results, title):
    print(f"\n{'═'*78}")
    print(f"  {title}")
    print(f"{'═'*78}")
    header = f"{'Metric':<14} {'no_net':>8} {'with_net':>9} {'Δ (with−no)':>12} {'t':>7} {'p':>8}  sig?"
    print(header)
    print("─" * 78)
    for r in results:
        flag  = "✓ YES" if r["sig"] else "  no"
        delta_sign = "+" if r["diff"] >= 0 else ""
        print(
            f"{r['metric']:<14}"
            f" {r['mean_no']:>8.4f}"
            f" {r['mean_with']:>9.4f}"
            f" {delta_sign}{r['diff']:>+10.4f}"
            f" {r['t']:>7.3f}"
            f" {r['p']:>8.4f}"
            f"  {flag}"
        )
    print("─" * 78)
    n_sig = sum(r["sig"] for r in results)
    print(f"  Significant at α={ALPHA}: {n_sig}/{len(results)} metrics")

for split in ("val", "test"):
    print_table(run_tests(lr_no, lr_with, split), f"XGBoost — {split.upper()}")

print("\n\nEffect sizes (Cohen's d) for significant results:")
print(f"{'Model':<14} {'Split':<6} {'Metric':<14} {'Cohen d':>9}  interpretation")
print("─" * 65)

def cohens_d(no_data, with_data, split, metric):
    key = f"per_seed_{split}"
    a = extract(no_data[key], metric)
    b = extract(with_data[key], metric)
    diff = b - a
    return diff.mean() / diff.std(ddof=1)

def interpret(d):
    ad = abs(d)
    if ad < 0.2:  return "negligible"
    if ad < 0.5:  return "small"
    if ad < 0.8:  return "medium"
    return "large"

for model_name, no_data, with_data in [
    ("XGBoost", lr_no, lr_with),
]:
    for split in ("val", "test"):
        res = run_tests(no_data, with_data, split)
        for r in res:
            if r["sig"]:
                d = cohens_d(no_data, with_data, split, r["metric"])
                print(f"{model_name:<14} {split:<6} {r['metric']:<14} {d:>9.3f}  {interpret(d)}")


══════════════════════════════════════════════════════════════════════════════
  XGBoost — VAL
══════════════════════════════════════════════════════════════════════════════
Metric           no_net  with_net  Δ (with−no)       t        p  sig?
──────────────────────────────────────────────────────────────────────────────
HitRate@1        0.7744    0.9949 +   +0.2205 384.457   0.0000  ✓ YES
HitRate@5        0.8625    0.9970 +   +0.1345  80.937   0.0002  ✓ YES
HitRate@10       0.9002    0.9977 +   +0.0974  60.909   0.0003  ✓ YES
HitRate@20       0.9399    0.9984 +   +0.0585  57.018   0.0003  ✓ YES
NDCG@1           0.7744    0.9949 +   +0.2205 384.457   0.0000  ✓ YES
NDCG@5           0.8214    0.9960 +   +0.1746 267.830   0.0000  ✓ YES
NDCG@10          0.8336    0.9962 +   +0.1627 452.135   0.0000  ✓ YES
NDCG@20          0.8436    0.9964 +   +0.1528 613.490   0.0000  ✓ YES
MRR              0.8172    0.9959 +   +0.1787 660.945   0.0000  ✓ YES
──────────────────────────────────────────────